# cudf-bench: Route B prototype (Step 4)

**Runtime → T4 GPU, Run all.** ~8 min.

Heavy-hitter-aware groupby (`benchmark/routeb.py`): hot keys aggregate in per-block shared memory, tail keys in a dense global table — what a frequency-aware gate in libcudf could do (see rapidsai/cudf#23256). Correctness is asserted against cuDF's own result before any timing.

Honest scorecard requires both: **win on skew, no big loss on uniform.**

In [ ]:
!nvidia-smi -L
!nvidia-smi -lgc 1590,1590 || echo 'clock lock failed - numbers will be noisier'

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
%cd /content
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd /content/cudf-bench
!git fetch -q && git reset -q --hard origin/main && git clean -qfd results
!rm -f results/routeb.csv
%pip install -q -e .

In [ ]:
!python -m benchmark.routeb --skew 1.5 --iters 8 --out results/routeb.csv
!python -m benchmark.routeb --skew 2.0 --iters 8 --out results/routeb.csv
!python -m benchmark.routeb --skew 0   --iters 8 --out results/routeb.csv
!nvidia-smi -rgc || true

In [ ]:
import pandas as pd
df = pd.read_csv("results/routeb.csv")
print(df.pivot_table(index="skew", columns="method", values="time_s", aggfunc="median").round(5))

In [ ]:
from google.colab import files
files.download("/content/cudf-bench/results/routeb.csv")